# Colab 11b — Iter 2: breaking the synthetic-shortcut

**Continuation of colab11.** Iter 1 trained successfully on synthetic V1 (Pearson 0.72) but **failed on natural V1 (Pearson −0.22)**. This notebook documents what went wrong, applies three concrete fixes, and re-runs the experiment.

## Iter 1 finding — the network learned generative-procedure structure, not edit distance

Iter 1 mixed two pair sources:
- **Natural pairs** from `cath_s20_pairs_sample.csv.gz`: different CATH proteins, `aa_score` labels in `[0.01, 0.33]`.
- **Synthetic perturbation pairs**: one sequence is the other with k random sub/ins/del. Computed `normLev` labels in `[0.4, 1.0]`.

The two label distributions were **disjoint**. The two pair sources also produced visibly different *input* structure:
- Synthetic pairs share long contiguous identical substrings (perturbation only edits k of N positions).
- Natural pairs share no contiguous structure (different proteins, ≤20% sequence identity by construction in CATH-S20).

**The model exploited this as a shortcut.** It learned a two-step decision rule rather than the Levenshtein function:
1. *Classify* the input pair: synthetic-procedure or natural-procedure?
2. *Emit the cluster mean* of the corresponding label distribution (~0.2 for natural, ~0.55 for synthetic), with weak refinement within each cluster.

Evidence (iter 1 V1 metrics):

| Held-out set | True label range | Pearson r | Pattern |
|---|---|---|---|
| Synthetic | [0.4, 1.0] | 0.722 | works (within-cluster refinement) |
| Natural | [0.01, 0.33] | **−0.22** | flat at ~0.27 — no refinement |
| High-sim natural | [0.30, 0.87] | 0.108 | flat at ~0.30–0.50 |

## Why this is shortcut learning

This is the classical phenomenon described by Geirhos et al. 2020 (*Shortcut Learning in Deep Neural Networks*, Nature Machine Intelligence): when a feature is correlated with the label *in the training distribution* but absent or anti-correlated *outside* it, the network preferentially learns the spurious feature because it's easier to extract. Here:

- **Wanted feature:** alignment-like global comparison → approximates `1 − lev/max(L)`, generalizes across pair types.
- **Shortcut feature:** *generative-procedure structure* — does the pair share long contiguous identical substrings? Trivially detectable by local pattern matching, perfectly correlated with the label in training, useless outside training (real biological pairs never have this property by construction).

Reframing in domain language: the network *is* learning a kind of structural relationship between the two inputs — but it's the structure of **how we generated the pair** (perturbation vs independent draw), not the structure of **biological similarity**. The label happened to align with the generative procedure during training, so the shortcut worked for the loss but not for the function we claimed to approximate.

## Iter 2 fixes (this notebook)

**Fix A — Add natural pairs in [0.30, 0.87] to training.** Use `cath_s20_pairs_high.csv.gz` filtered to train70 IDs. These are *natural* pairs (different CATH proteins, real biology) covering exactly the label range that synthetic pairs were filling. The natural/synthetic dichotomy stops aligning with the label distribution → the shortcut feature loses its predictive power on training loss.

**Fix B — Reduce synthetic share.** Drop synthetic from 30K to 10K pairs. Synthetic stays useful for the very high-similarity tail (≥0.7) which natural data does not supply, but no longer dominates the loss landscape.

**Fix C — Stratified V2 sampling.** Iter 1 V2 returned only 19 same-SF pairs in 5K random samples (test30 has 1,246 superfamilies → P(same-SF) ≈ 0.08%). Replace random sampling with: balanced sample of N same-SF + N different-SF pairs.

Architecture, loss (plain MSE), and optimizer are unchanged from colab11.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score
from collections import defaultdict

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN_KEEP = 50
MAX_LEN_KEEP = 200
MAX_LEN = 200

# Iter 2 pair counts — natural-high replaces some synthetic
N_NATURAL_LOW_TRAIN  = 30000   # capped by availability after filtering
N_NATURAL_HIGH_TRAIN = 10000   # capped by availability
N_SYNTH_TRAIN        = 10000   # reduced from 30000
N_NATURAL_LOW_HELD   = 5000
N_NATURAL_HIGH_HELD  = 5000    # capped by availability (was 294)
N_SYNTH_HELD         = 5000

# V2 stratified sampling
N_V2_SAME = 2500
N_V2_DIFF = 2500

print(f'Vocab {VOCAB_SIZE}, PAD {PAD_IDX}, length [{MIN_LEN_KEEP},{MAX_LEN_KEEP}], pad to {MAX_LEN}')

## 3. Helpers

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - levenshtein(a, b) / L

def is_standard(seq):
    return all(c in AA_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

## 4. Load proteins

In [ ]:
def load_protein_csv(path):
    df = pd.read_csv(path, compression='gzip').dropna(subset=['aa_seq', 'SuperFamily'])
    df['aa_seq'] = df['aa_seq'].astype(str)
    df = df[df['aa_seq'].str.len().between(MIN_LEN_KEEP, MAX_LEN_KEEP)]
    df = df[df['aa_seq'].apply(is_standard)]
    return df

train_df = load_protein_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = load_protein_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))

train_seqs = {r['domain_id']: r['aa_seq'] for _, r in train_df.iterrows()}
train_sf   = {r['domain_id']: r['SuperFamily'] for _, r in train_df.iterrows()}
test_seqs  = {r['domain_id']: r['aa_seq'] for _, r in test_df.iterrows()}
test_sf    = {r['domain_id']: r['SuperFamily'] for _, r in test_df.iterrows()}

print(f'train: {len(train_seqs)} proteins, {len(set(train_sf.values()))} superfamilies')
print(f'test:  {len(test_seqs)} proteins, {len(set(test_sf.values()))} superfamilies')

## 5. Load pair files (both headerless)

In [ ]:
PAIR_COLS = ['domain_p2', 'domain_p1', 'ss_score', 'aa_score',
             'TM_min', 'TM_max', 'cath_superFamily']

def load_pairs(path):
    df = pd.read_csv(path, compression='gzip', header=None, names=PAIR_COLS)
    df['aa_score'] = pd.to_numeric(df['aa_score'], errors='coerce')
    return df.dropna(subset=['aa_score']).reset_index(drop=True)

pairs_sample = load_pairs(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'))
pairs_high   = load_pairs(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'))

print(f'pairs_sample: {len(pairs_sample)} rows, aa_score range [{pairs_sample.aa_score.min():.3f}, {pairs_sample.aa_score.max():.3f}]')
print(f'pairs_high:   {len(pairs_high)} rows, aa_score range [{pairs_high.aa_score.min():.3f}, {pairs_high.aa_score.max():.3f}]')

## 6. Filter pairs to train / test sets — including the new natural-high training source

In [ ]:
def filter_pairs(df, id_set):
    return df[df['domain_p1'].isin(id_set) & df['domain_p2'].isin(id_set)].reset_index(drop=True)

train_ids = set(train_seqs.keys())
test_ids  = set(test_seqs.keys())

natural_low_train  = filter_pairs(pairs_sample, train_ids)   # [~0.01, ~0.33]
natural_low_test   = filter_pairs(pairs_sample, test_ids)
natural_high_train = filter_pairs(pairs_high,   train_ids)   # [0.30, 0.87] — KEY for fix A
natural_high_test  = filter_pairs(pairs_high,   test_ids)

print(f'natural-low train pairs:  {len(natural_low_train)}')
print(f'natural-low test pairs:   {len(natural_low_test)}')
print(f'natural-high train pairs: {len(natural_high_train)}  ← new in iter 2')
print(f'natural-high test pairs:  {len(natural_high_test)}')

## 7. Build training pair list — three sources blended

Reduces the natural/synthetic dichotomy that the iter-1 model exploited as a shortcut. Natural pairs now span the full label range; synthetic only fills the very high tail.

In [ ]:
def perturb(seq, k, max_len=MAX_LEN):
    s = list(seq)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in AA_ALPHABET if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(list(AA_ALPHABET)))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def make_natural_pairs(natural_df, seq_dict, n_pairs):
    if len(natural_df) >= n_pairs:
        df = natural_df.sample(n=n_pairs, random_state=SEED).reset_index(drop=True)
    else:
        df = natural_df
    pairs = []
    for _, r in df.iterrows():
        a = seq_dict.get(r['domain_p1']); b = seq_dict.get(r['domain_p2'])
        if a and b:
            pairs.append((a, b, float(r['aa_score'])))
    return pairs

def make_synthetic_pairs(seq_dict, n_pairs):
    seeds = list(seq_dict.values())
    pairs = []
    for _ in range(n_pairs * 4):
        if len(pairs) >= n_pairs: break
        seed = seeds[int(rng.integers(0, len(seeds)))]
        k = int(rng.integers(1, max(2, int(0.8 * len(seed)))))
        other = perturb(seed, k)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

print('Building training pairs…')
tr_low  = make_natural_pairs(natural_low_train,  train_seqs, N_NATURAL_LOW_TRAIN)
tr_high = make_natural_pairs(natural_high_train, train_seqs, N_NATURAL_HIGH_TRAIN)
tr_synth = make_synthetic_pairs(train_seqs, N_SYNTH_TRAIN)
print(f'  natural-low:  {len(tr_low)}')
print(f'  natural-high: {len(tr_high)}')
print(f'  synthetic:    {len(tr_synth)}')

print('\nBuilding held-out pairs…')
ho_low  = make_natural_pairs(natural_low_test,  test_seqs, N_NATURAL_LOW_HELD)
ho_high = make_natural_pairs(natural_high_test, test_seqs, N_NATURAL_HIGH_HELD)
ho_synth = make_synthetic_pairs(test_seqs, N_SYNTH_HELD)
print(f'  natural-low:  {len(ho_low)}')
print(f'  natural-high: {len(ho_high)}')
print(f'  synthetic:    {len(ho_synth)}')

train_pairs = tr_low + tr_high + tr_synth
rng.shuffle(train_pairs)
print(f'\nTotal training pairs: {len(train_pairs)}')

plt.figure(figsize=(10, 3))
plt.hist([np.array([p[2] for p in tr_low]),
          np.array([p[2] for p in tr_high]),
          np.array([p[2] for p in tr_synth])],
         bins=40, stacked=True,
         label=[f'natural-low ({len(tr_low)})',
                f'natural-high ({len(tr_high)})',
                f'synthetic ({len(tr_synth)})'],
         edgecolor='k', alpha=0.75)
plt.xlabel('Label (normLev similarity)'); plt.ylabel('Count')
plt.title('Iter 2 training-pair label distribution')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 8. Dataset / DataLoader / Architecture (unchanged from colab11)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b, label = self.pairs[i]
        return encode_pad(a), encode_pad(b), torch.tensor(label, dtype=torch.float32)

train_dl = DataLoader(PairDataset(train_pairs), batch_size=128, shuffle=True)

class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 max_len=MAX_LEN, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2 * max_len, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x * mask.unsqueeze(1)
        x = x.flatten(1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)

class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__(); self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        ea, eb = self.encoder(a), self.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0
    def encode(self, x): return self.encoder(x)

model = SiameseNetwork().to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 9. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 30
losses = []
model.train()
for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0; nb = 0
    for a, b, label in train_dl:
        a = a.to(device); b = b.to(device); label = label.to(device)
        pred = model(a, b)
        loss = F.mse_loss(pred, label)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{num_epochs} — MSE: {avg:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Iter 2 training loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final MSE: {losses[-1]:.5f}')

## 10. V1 — three held-out scatters

If iter 2 worked, all three should show diagonal alignment instead of clustering at the cluster mean.

In [ ]:
def predict_pairs(pairs, batch_size=512):
    model.eval()
    true_v, pred_v = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            pred = model(a, b).cpu().numpy()
            true_v.extend([p[2] for p in batch])
            pred_v.extend(pred.tolist())
    return np.array(true_v), np.array(pred_v)

def report(name, true_v, pred_v):
    if len(true_v) < 10:
        print(f'{name}: n={len(true_v)} too few'); return None
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    print(f'{name:>22}: n={len(true_v):>5}  Pearson={pr:+.3f}  Spearman={sr:+.3f}')
    return pr, sr

true_synth, pred_synth = predict_pairs(ho_synth)
true_low,   pred_low   = predict_pairs(ho_low)
true_high,  pred_high  = predict_pairs(ho_high)

print('=== V1 metrics ===')
report('synthetic held-out',    true_synth, pred_synth)
report('natural-low held-out',  true_low,   pred_low)
report('natural-high held-out', true_high,  pred_high)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (true_v, pred_v, title) in zip(axes, [
    (true_synth, pred_synth, 'Synthetic held-out'),
    (true_low,   pred_low,   'Natural-low held-out'),
    (true_high,  pred_high,  'Natural-high held-out'),
]):
    if len(true_v) < 10:
        ax.set_title(f'{title}\n(too few pairs)'); continue
    ax.scatter(true_v, pred_v, alpha=0.15, s=6)
    ax.plot([0, 1], [0, 1], 'r-', linewidth=2, label='y = x')
    pr, _ = pearsonr(true_v, pred_v)
    sr, _ = spearmanr(true_v, pred_v)
    ax.set_title(f'{title}\nn={len(true_v)}  r={pr:+.3f}  ρ={sr:+.3f}')
    ax.set_xlabel('True normLev'); ax.set_ylabel('Predicted')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11. V2 — stratified same/different SuperFamily separation

Iter 1 V2 returned only 19 same-SF pairs in 5K random samples (test30 has 1,246 superfamilies → P(same-SF) ≈ 0.08%). Iter 2 explicitly samples balanced same-SF and different-SF pairs.

In [ ]:
# Group test30 proteins by superfamily
sf_to_ids = defaultdict(list)
for did, sf in test_sf.items():
    if did in test_seqs:
        sf_to_ids[sf].append(did)
multi_sf = [sf for sf, ids in sf_to_ids.items() if len(ids) >= 2]
print(f'Test30 superfamilies with ≥2 proteins: {len(multi_sf)} / {len(sf_to_ids)}')

# Sample same-SF pairs
same_pairs = []
while len(same_pairs) < N_V2_SAME:
    sf = multi_sf[int(rng.integers(0, len(multi_sf)))]
    ids = sf_to_ids[sf]
    i, j = rng.integers(0, len(ids), size=2)
    if i == j: continue
    same_pairs.append((test_seqs[ids[i]], test_seqs[ids[j]], 0.0))

# Sample different-SF pairs
test_id_list = list(test_seqs.keys())
diff_pairs = []
while len(diff_pairs) < N_V2_DIFF:
    i, j = rng.integers(0, len(test_id_list), size=2)
    if i == j: continue
    id_a, id_b = test_id_list[i], test_id_list[j]
    if test_sf[id_a] == test_sf[id_b]: continue
    diff_pairs.append((test_seqs[id_a], test_seqs[id_b], 0.0))

v2_pairs = same_pairs + diff_pairs
v2_labels = np.array([1] * len(same_pairs) + [0] * len(diff_pairs))
_, v2_pred = predict_pairs(v2_pairs)

auroc = roc_auc_score(v2_labels, v2_pred)
print(f'V2 AUROC (stratified, same={len(same_pairs)}, diff={len(diff_pairs)}): {auroc:.3f}')

plt.figure(figsize=(10, 5))
plt.hist(v2_pred[v2_labels == 0], bins=40, alpha=0.6, label=f'different SF (n={len(diff_pairs)})', edgecolor='k')
plt.hist(v2_pred[v2_labels == 1], bins=40, alpha=0.6, label=f'same SF (n={len(same_pairs)})', edgecolor='k')
plt.xlabel('Predicted similarity'); plt.ylabel('Count')
plt.title(f'V2: predicted similarity by SuperFamily class — AUROC={auroc:.3f}')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 12. Iter 2 success criteria

Compared to iter 1 baseline:

| Metric | Iter 1 | Iter 2 target | Why |
|---|---|---|---|
| Synthetic V1 r | 0.722 | **≥ 0.65** (allow some loss — synthetic share is now 16% of training, was 50%) | synthetic eval is now an out-of-distribution-shift test |
| Natural-low V1 r | **−0.22** | **≥ +0.50** | this was the iter 1 failure — must recover |
| Natural-high V1 r | 0.108 | **≥ +0.40** | should now show diagonal in [0.30, 0.87] |
| V2 AUROC | 0.675 (n_same=19, unstable) | **≥ 0.70** with n_same=2500 (stable) | honest measurement |

**If natural-low V1 is now positive,** the shortcut has been broken — natural pairs spanning the full label range force the network to actually compute a similarity function instead of classifying pair source.

**If natural-low V1 is still flat,** something else is going on — possibly the encoder is too small for variable-length real proteins, or natural-low pairs fundamentally don't carry enough signal in the [0.01, 0.33] band to discriminate. We'd then increase encoder capacity (`embed_dim=64, out_dim=256`) before changing the training data again.

**Phase 5 (cross-representation) gates on iter 2 passing.** If V1 looks broken on AA, no point running it on 3Di yet.